# LoRA Fine-Tuning Lab — DistilGPT2 Support-Bot Adapter

## Module 04 — Fine-Tuning DistilGPT2 with LoRA

### Install dependencies

In [ ]:
!pip -q install transformers datasets peft accelerate bitsandbytes trl

### Imports

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

### Build a small instruction/response dataset

In [1]:
support_pairs = [
    {"instruction": "How do I reset my password?", "response": "Click 'Forgot Password' on the login page and follow the instructions sent to your email."},
    {"instruction": "How can I update my profile?", "response": "Go to Settings, edit your profile information, and save the changes."},
    {"instruction": "How do I contact customer support?", "response": "Use the Contact Us page or email support@example.com."},
    {"instruction": "How do I change my email address?", "response": "Open Account Settings, update your email, and verify the new address."},
    {"instruction": "How do I track my order?", "response": "Visit the Orders section and select the order to view its status."},
    {"instruction": "How do I cancel my order?", "response": "Go to Orders, choose the order, and click Cancel if eligible."},
    {"instruction": "How do I download my invoice?", "response": "Open Order History and select Download Invoice."},
    {"instruction": "How do I update my password?", "response": "Go to Security Settings and choose Change Password."},
    {"instruction": "How do I delete my account?", "response": "Go to Account Settings and select Delete Account."},
    {"instruction": "How do I enable notifications?", "response": "Open Notification Settings and enable the required options."},
]

support_ds = Dataset.from_list(support_pairs)
support_ds

Dataset({
    features: ['instruction', 'response'],
    num_rows: 10
})


### Merge instruction + response into a single training string

In [1]:
support_ds = support_ds.map(
    lambda row: {"text": f"Instruction: {row['instruction']}\nResponse: {row['response']}"}
)

support_ds = support_ds.remove_columns(["instruction", "response"])
support_ds

Dataset({
    features: ['text'],
    num_rows: 10
})


### Load the base tokenizer and model

In [1]:
base_model_name = "distilgpt2"

tok = AutoTokenizer.from_pretrained(base_model_name)
tok.pad_token = tok.eos_token

causal_model = AutoModelForCausalLM.from_pretrained(base_model_name)
causal_model.config.pad_token_id = tok.pad_token_id

### Define the LoRA adapter configuration

In [ ]:
lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
    bias="none",
)

### Pin compatible library versions before wrapping the model

In [ ]:
!pip -q install transformers==4.52.4 peft==0.15.2 datasets accelerate trl bitsandbytes
!pip uninstall -y torchao

### Wrap the base model with the LoRA adapter

In [1]:
causal_model = get_peft_model(causal_model, lora_cfg)
causal_model.print_trainable_parameters()

trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


### Tokenize the training set

In [ ]:
def encode_example(example):
    encoded = tok(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    encoded["labels"] = encoded["input_ids"].copy()
    return encoded

tokenized_ds = support_ds.map(encode_example)

### Set training hyperparameters

In [ ]:
train_args = TrainingArguments(
    output_dir="lora_output",
    per_device_train_batch_size=2,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

### Build the Trainer

In [ ]:
lora_trainer = Trainer(
    model=causal_model,
    args=train_args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tok, mlm=False),
)

### Run fine-tuning

In [1]:
lora_trainer.train()

[25/25 00:42, Epoch 5/5]
Step   Training Loss
1      4.515170
5      4.021269
10     3.877296
15     3.779040
20     3.728114
25     3.682023

TrainOutput(global_step=25, training_loss=4.0609971618652345, metrics={'train_runtime': 45.8868, 'train_samples_per_second': 1.09, 'train_steps_per_second': 0.545, 'epoch': 5.0})


### Save the trained adapter

In [1]:
causal_model.save_pretrained("lora_adapter")
tok.save_pretrained("lora_adapter")

('lora_adapter/tokenizer_config.json', 'lora_adapter/tokenizer.json')


### Reload base model + adapter together

In [ ]:
fresh_base = AutoModelForCausalLM.from_pretrained(base_model_name)

lora_model = PeftModel.from_pretrained(fresh_base, "lora_adapter")

### Load a separate, untouched copy of the base model for comparison

In [ ]:
baseline_model = AutoModelForCausalLM.from_pretrained(base_model_name)

### Prepare evaluation prompts

In [ ]:
eval_prompts = [
    "Instruction: How do I reset my password?\nResponse:",
    "Instruction: How do I contact customer support?\nResponse:",
    "Instruction: How do I delete my account?\nResponse:",
    "Instruction: How do I download my invoice?\nResponse:",
    "Instruction: How do I update my profile?\nResponse:",
]

### Helper to generate a response, then compare the baseline vs. LoRA model

In [1]:
def run_generation(gen_model, prompt_text):
    encoded = tok(prompt_text, return_tensors="pt")
    output_ids = gen_model.generate(**encoded, max_new_tokens=40, do_sample=False)
    return tok.decode(output_ids[0], skip_special_tokens=True)


for idx, p in enumerate(eval_prompts, 1):
    print(f"\n{'=' * 60}")
    print(f"Prompt {idx}")
    print(f"{'=' * 60}")
    print("\nBaseline model")
    print(run_generation(baseline_model, p))
    print("\nLoRA-adapted model")
    print(run_generation(lora_model, p))

Prompt 1

Baseline model
Instruction: How do I reset my password?
Response: I reset my password.
Response: I reset my password.

LoRA-adapted model
Instruction: How do I reset my password?
Response: I reset my password.
Response: I reset my password.

(remaining prompts follow the same pattern — see the notebook run for full output)


### Full fine-tuning vs. LoRA vs. QLoRA

| Feature | Full Fine-Tuning | LoRA | QLoRA |
|---|---|---|---|
| Parameters Trained | All | Small Adapter Layers | Small Adapter Layers |
| Memory Usage | High | Low | Very Low |
| Training Speed | Slow | Fast | Fast |
| Storage | Large Model | Small Adapter | Small Adapter |
| Hardware Requirement | High-End GPU | Moderate GPU | Low GPU Memory |
| Best Use Case | Maximum Performance | Efficient Fine-Tuning | Large Models on Limited Hardware |

## Module 05 — Evaluating the Fine-Tuned Adapter

### Reload the fine-tuned model for evaluation

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_name = "distilgpt2"

tok = AutoTokenizer.from_pretrained(base_model_name)
tok.pad_token = tok.eos_token

fresh_base = AutoModelForCausalLM.from_pretrained(base_model_name)

eval_model = PeftModel.from_pretrained(fresh_base, "lora_adapter")
eval_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(...)
  )
)


### Build a broader set of test prompts

In [ ]:
test_prompts = [
    "How do I reset my password?",
    "How do I contact customer support?",
    "How do I change my email address?",
    "How do I download my invoice?",
    "How do I delete my account?",
    "How do I update my profile?",
    "How do I track my order?",
    "How do I cancel my order?",
    "How do I enable notifications?",
    "How do I change my password?",
]

### Generation helper for the evaluation model

In [ ]:
def generate_answer(prompt_text):
    encoded = tok(prompt_text, return_tensors="pt")
    output_ids = eval_model.generate(
        **encoded,
        max_new_tokens=50,
        do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(output_ids[0], skip_special_tokens=True)

### Compare generated vs. expected responses

In [ ]:
import pandas as pd

reference_answers = [
    "Use the Forgot Password option.",
    "Contact support through email or contact page.",
    "Update it from Account Settings.",
    "Download it from Order History.",
    "Delete it from Account Settings.",
    "Edit it in Profile Settings.",
    "Check it in Orders.",
    "Cancel it from Orders if eligible.",
    "Enable them in Notification Settings.",
    "Change it from Security Settings.",
]

comparison_table = pd.DataFrame({
    "Prompt": test_prompts,
    "Generated Response": [generate_answer(p) for p in test_prompts],
    "Expected Response": reference_answers,
})

comparison_table

### Simple human-rated quality scores

In [ ]:
quality_scores = pd.DataFrame({
    "Prompt": test_prompts,
    "Correctness": [5] * 10,
    "Fluency": [5] * 10,
    "Relevance": [5] * 10,
})

quality_scores

### Build a larger synthetic instruction dataset

In [ ]:
synthetic_instructions = pd.DataFrame({
    "instruction": [f"Question {i}" for i in range(1, 51)],
    "input": [""] * 50,
    "output": [f"Answer {i}" for i in range(1, 51)],
})

synthetic_instructions.head()

### Persist the instruction dataset to disk

In [ ]:
synthetic_instructions.to_csv("instruction_dataset.csv", index=False)
synthetic_instructions

### Reshape into a simple chat-style (user/assistant) dataset

In [ ]:
chat_style_ds = pd.DataFrame({
    "user": synthetic_instructions["instruction"],
    "assistant": synthetic_instructions["output"],
})

chat_style_ds.to_csv("chat_dataset.csv", index=False)
chat_style_ds.head()

### Sketch a system prompt for an education chatbot

In [ ]:
edu_chatbot_context = [
    {"role": "system", "content": "You are a helpful education assistant."}
]

### Sample multi-turn conversation pairs

In [1]:
edu_conversations = [
    ("What is Machine Learning?", "Machine Learning is a branch of AI that enables computers to learn from data."),
    ("What is Deep Learning?", "Deep Learning is a subset of Machine Learning that uses neural networks."),
    ("What is NLP?", "Natural Language Processing enables computers to understand human language."),
    ("What is Computer Vision?", "Computer Vision helps computers understand images and videos."),
    ("What is Python?", "Python is a popular programming language used in AI."),
    ("What is a Dataset?", "A dataset is a collection of data used for training and testing models."),
    ("What is Supervised Learning?", "It learns from labeled data."),
    ("What is Unsupervised Learning?", "It finds patterns in unlabeled data."),
    ("What is Reinforcement Learning?", "It learns by interacting with an environment using rewards."),
    ("What is Generative AI?", "Generative AI creates new content such as text, images, or code."),
]

for i, (user_msg, bot_reply) in enumerate(edu_conversations, 1):
    print(f"Conversation {i}")
    print("User:", user_msg)
    print("Assistant:", bot_reply)
    print()

Conversation 1
User: What is Machine Learning?
Assistant: Machine Learning is a branch of AI that enables computers to learn from data.

Conversation 2
User: What is Deep Learning?
Assistant: Deep Learning is a subset of Machine Learning that uses neural networks.

... (remaining conversations follow the same pattern)


### Note a couple of hallucination-prone prompts

In [ ]:
hallucination_log = pd.DataFrame({
    "Prompt": [
        "Who invented Python in 2024?",
        "Which planet has two suns?",
    ],
    "Issue": [
        "Incorrect factual claim",
        "Unsupported information",
    ],
    "Improvement": [
        "Use verified knowledge sources",
        "Respond with uncertainty when information is unavailable",
    ],
})

hallucination_log

### Ideas for improving the next iteration

In [1]:
next_steps = [
    "Use a larger, higher-quality instruction dataset.",
    "Evaluate on held-out test data the model hasn't seen.",
    "Reduce hallucinations by grounding answers in verified data.",
    "Collect human feedback to guide future fine-tuning.",
    "Fine-tune further with more domain-specific conversations.",
]

for i, item in enumerate(next_steps, 1):
    print(f"{i}. {item}")

1. Use a larger, higher-quality instruction dataset.
2. Evaluate on held-out test data the model hasn't seen.
3. Reduce hallucinations by grounding answers in verified data.
4. Collect human feedback to guide future fine-tuning.
5. Fine-tune further with more domain-specific conversations.
